In [1]:
import pandas as pd

In [5]:
# load file with ALCC file names
df = pd.read_csv("datasets/ALCC/ALCC_output_October2026/file_list_fromNERSC_fileCheck_v2.csv")

print(df.columns)

Index(['folder', 'file', 'mode', 'TC run', 'year', 'month', 'full_path'], dtype='str')


In [6]:
print("Modes:")
print(sorted(df["mode"].unique()))

print("\nRuns:")
print(sorted(df["TC run"].unique()))

print("\nYears:")
print(sorted(df["year"].unique()))

print("\nMonths:")
print(sorted(df["month"].unique()))

print("\nNumber of rows:", len(df))

Modes:
['nn', 'no', 'np', 'on', 'oo', 'op', 'pn', 'po', 'pp']

Runs:
[np.int64(1), np.int64(2)]

Years:
[np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013)]

Months:
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', 'ta']

Number of rows: 8272


In [12]:
# df.groupby(["mode", "TC run"]).size()

duplicates = (
    df.groupby(["mode", "TC run", "year", "month"])
      .size()
      .reset_index(name="count")
)

duplicates = duplicates[duplicates["count"] > 1]

print(f"Number of duplicated year/month combinations: {len(duplicates)}")

duplicates

Number of duplicated year/month combinations: 2494


,mode,TC run,year,month,count
0,nn,1,1984,07,2
72,nn,1,1990,07,2
144,nn,1,1996,07,2
216,nn,1,2002,07,2
288,nn,1,2008,07,2
...,...,...,...,...,...
5673,pp,1,2012,11,2
5674,pp,1,2012,12,2
5675,pp,1,2013,01,2
5676,pp,1,2013,02,2


In [15]:
unique_counts = (
    df.groupby(["mode", "TC run"])
      .apply(lambda x: x[["year", "month"]].drop_duplicates().shape[0])
)

print(unique_counts)

mode  TC run
nn    1         348
      2         351
no    1         354
      2         354
np    1         354
      2         348
on    1         342
      2         354
oo    1         354
      2         354
op    1         345
      2         348
pn    1         354
      2          48
po    1         354
      2         371
pp    1         345
      2          30
dtype: int64


In [18]:
# Make sure year and month are numeric
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["month"] = pd.to_numeric(df["month"], errors="coerce")

# Check the data types
print(df[["year", "month"]].dtypes)

# Check whether there are any missing months
print("Missing months:", df["month"].isna().sum())

# Convert month to integer while preserving missing values
df["month"] = df["month"].astype("Int64")

print(df[["year", "month"]].dtypes)

year       int64
month    float64
dtype: object
Missing months: 89
year     int64
month    Int64
dtype: object


In [20]:
import itertools
import pandas as pd

modes = sorted(df["mode"].unique())
runs = [1, 2]
years = range(1985, 2015)
months = range(1, 13)

expected = pd.DataFrame(
    itertools.product(modes, runs, years, months),
    columns=["mode", "TC run", "year", "month"]
)

# Match the types
expected["year"] = expected["year"].astype("int64")
expected["month"] = expected["month"].astype("Int64")

actual = df[["mode", "TC run", "year", "month"]].drop_duplicates()

missing = expected.merge(
    actual,
    on=["mode", "TC run", "year", "month"],
    how="left",
    indicator=True
)

missing = missing[missing["_merge"] == "left_only"].drop(columns="_merge")

print(f"Expected combinations: {len(expected):,}")
print(f"Actual combinations:   {len(actual):,}")
print(f"Missing combinations:  {len(missing):,}")

display(
    missing.sort_values(["mode", "TC run", "year", "month"])
)

Expected combinations: 6,480
Actual combinations:   5,708
Missing combinations:  963


,mode,TC run,year,month
342,nn,1,2013,7
343,nn,1,2013,8
344,nn,1,2013,9
345,nn,1,2013,10
346,nn,1,2013,11
...,...,...,...,...
6475,pp,2,2014,8
6476,pp,2,2014,9
6477,pp,2,2014,10
6478,pp,2,2014,11


In [24]:
# missing combos
missing_summary = (
    missing
    .groupby(["mode", "TC run"])
    .size()
    .reset_index(name="missing")
)

# display(missing_summary)

# Ignore the two known-incomplete combinations
missing_relevant = missing[
    ~(
        ((missing["mode"] == "pn") & (missing["TC run"] == 2)) |
        ((missing["mode"] == "pp") & (missing["TC run"] == 2))
    )
].copy()

display(
    missing_relevant.sort_values(
        ["mode", "TC run", "year", "month"]
    )
)

# export to csv
missing_relevant.to_csv("datasets/ALCC/ALCC_output_October2026/missing_files.csv")

,mode,TC run,year,month
342,nn,1,2013,7
343,nn,1,2013,8
344,nn,1,2013,9
345,nn,1,2013,10
346,nn,1,2013,11
...,...,...,...,...
6115,pp,1,2014,8
6116,pp,1,2014,9
6117,pp,1,2014,10
6118,pp,1,2014,11


In [27]:
# check duplicates
duplicates = (
    df.groupby(["mode", "TC run", "year", "month"])
      .size()
      .reset_index(name="file_count")
)

duplicates = duplicates[duplicates["file_count"] > 1]

duplicate_files = df.merge(
    duplicates[["mode", "TC run", "year", "month"]],
    on=["mode", "TC run", "year", "month"],
    how="inner"
)

display(
    duplicate_files[
        ["folder", "file", "mode", "TC run", "year", "month"]
    ].sort_values(
        ["mode", "TC run", "year", "month", "file"]
    )
)

# save to csv
duplicate_files.to_csv("datasets/ALCC/ALCC_output_October2026/duplicate_files.csv")

,folder,file,mode,TC run,year,month
2570,ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001,ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001.eam.h...,nn,1,1984,7
2567,ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001,ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001.eam.h...,nn,1,1984,7
2566,ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001,ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001.eam.h...,nn,1,1990,7
2569,ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001,ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001.eam.h...,nn,1,1990,7
2571,ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001,ALCC.F20TR.ne0np4alcc30x4.hist.EOFnn.001.eam.h...,nn,1,1996,7
...,...,...,...,...,...,...
3787,ALCC.F20TR.ne0np4alcc30x4.hist.pp.001,ALCC.F20TR.ne0np4alcc30x4.hist.EOFpp.001.eam.h...,pp,1,2013,1
1100,ALCC.F20TR.ne0np4alcc30x4.hist.EOFpp.001,ALCC.F20TR.ne0np4alcc30x4.hist.EOFpp.001.eam.h...,pp,1,2013,2
3650,ALCC.F20TR.ne0np4alcc30x4.hist.pp.001,ALCC.F20TR.ne0np4alcc30x4.hist.EOFpp.001.eam.h...,pp,1,2013,2
1105,ALCC.F20TR.ne0np4alcc30x4.hist.EOFpp.001,ALCC.F20TR.ne0np4alcc30x4.hist.EOFpp.001.eam.h...,pp,1,2013,3


In [ ]:
from pathlib import Path
import pandas as pd

base_dir = Path("/global/cfs/projectdirs/m2637/ALCC/")

files = []

for file in base_dir.rglob("*"):
    if file.is_file():
        files.append({
            "folder": file.parent.name,
            "file": file.name,
            "full_path": str(file),
        })

df = pd.DataFrame(files)

output_file = base_dir / "file_list.csv"
df.to_csv(output_file, index=False)

print(f"Found {len(df)} files")
print(f"Saved to: {output_file}")

df.head(20)

